# 2D Chess Detection — Colab Training

End-to-end on Colab: clone the repo → generate synthetic data → train YOLO11m board detector + EfficientNetV2-S piece classifier → run on a chess-book PDF.

**Before running:** `Runtime → Change runtime type → T4 GPU` (or better).

## 0. Sanity check — GPU

In [ ]:
!nvidia-smi

## 1. Clone the repo

If your repo is private, set `GIT_TOKEN` via `userdata.set('GIT_TOKEN', '...')` first or just upload the source manually.

In [ ]:
REPO_URL = 'https://github.com/gsdt/2d-chess-detection.git'
BRANCH = 'main'

import os, subprocess, pathlib
os.chdir('/content')
if not pathlib.Path('2d-chess-detection').exists():
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL], check=True)
os.chdir('/content/2d-chess-detection')
!pwd && git log -1 --oneline

## 2. Install dependencies

Colab already ships with torch + cv2 + numpy. We only need to add the chess-specific bits.

In [ ]:
!pip install -q python-chess pypdfium2 cairosvg ultralytics tqdm

In [ ]:
import torch, chess, cv2, ultralytics
print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
print('cv2', cv2.__version__, 'chess', chess.__version__, 'ultralytics', ultralytics.__version__)

## 3. Generate synthetic datasets

These run on CPU but parallelize trivially across cores. With a free Colab CPU expect a few minutes for the sizes below; bump up if you have time.

In [ ]:
DET_PAGES = 1500   # synthetic book pages for YOLO
CLF_BOARDS = 3000  # synthetic boards → ~64 crops each → ~192k crops

!python -m src.synth.gen_detector --pages {DET_PAGES} --out data/detector
!python -m src.synth.gen_classifier --boards {CLF_BOARDS} --out data/classifier

In [ ]:
# Peek a sample to verify augmentation looks right.
import glob
from IPython.display import Image, display
page = sorted(glob.glob('data/detector/images/train/*.jpg'))[0]
print(page)
display(Image(filename=page, width=600))

## 4. Train YOLO11m board detector

In [ ]:
YOLO_EPOCHS = 30
YOLO_IMGSZ = 960
YOLO_BATCH = 16   # T4 → 16 OK at 960; bump to 32 on A100

!python -m src.train.train_yolo \
    --data data/detector/data.yaml \
    --epochs {YOLO_EPOCHS} --imgsz {YOLO_IMGSZ} --batch {YOLO_BATCH} \
    --device 0 --name board_detector

In [ ]:
# YOLO writes a results.png with PR / loss curves — show it.
from IPython.display import Image, display
import glob
for p in sorted(glob.glob('data/checkpoints/yolo_runs/board_detector/*.png'))[:3]:
    print(p); display(Image(filename=p, width=700))

## 5. Train piece classifier

In [ ]:
CLF_EPOCHS = 10
CLF_BATCH = 256   # T4 16GB; reduce if OOM

!python -m src.train.train_classifier \
    --epochs {CLF_EPOCHS} --img-size 96 --batch-size {CLF_BATCH} --workers 4 \
    --data data/classifier --out data/checkpoints/classifier.pt

## 6. Upload the PDF and run end-to-end inference

Two options — pick one of the cells below. **A) Upload from your laptop.** **B) Read from Google Drive.**

In [ ]:
# A) Upload from laptop
from google.colab import files
uploaded = files.upload()
PDF_PATH = '/content/2d-chess-detection/' + next(iter(uploaded))
print('PDF:', PDF_PATH)

In [ ]:
# B) Or mount Google Drive and read from there (skip if you used A)
# from google.colab import drive
# drive.mount('/content/drive')
# PDF_PATH = '/content/drive/MyDrive/Step 2 Extra.pdf'

In [ ]:
# Run on a small page range first to sanity-check, then expand.
PAGE_FROM = 1
PAGE_TO = 6   # exclusive

!python -m src.pipeline.run \
    --pdf "{PDF_PATH}" \
    --detector-weights data/checkpoints/yolo_runs/board_detector/weights/best.pt \
    --classifier-weights data/checkpoints/classifier.pt \
    --pages {PAGE_FROM} {PAGE_TO} \
    --device cuda \
    --out data/output

## 7. Inspect results

In [ ]:
import json, glob, pathlib
from IPython.display import Image, display

stem = pathlib.Path(PDF_PATH).stem
out_root = pathlib.Path('data/output') / stem
results = json.loads((out_root / 'results.json').read_text())
print(f'pages: {len(results)}  total boards: {sum(len(p["boards"]) for p in results)}')
for p in results[:2]:
    for b in p['boards']:
        print(f'page {p["page"]} board {b["index"]} conf={b["yolo_conf"]:.2f}  fen={b["fen"]}')

In [ ]:
# Show a few overlays + their predicted FEN renderings side by side.
from PIL import Image as PILImage
import io, glob
from IPython.display import display, HTML

for overlay in sorted(glob.glob(str(out_root / '*overlay.jpg')))[:2]:
    print(overlay)
    display(Image(filename=overlay, width=600))
for board in sorted(glob.glob(str(out_root / '*board_*[0-9].jpg')))[:6]:
    pred = board.replace('.jpg', '_pred.png')
    print(board, '→', pathlib.Path(pred).name)
    display(Image(filename=board, width=240))
    if pathlib.Path(pred).exists():
        display(Image(filename=pred, width=240))

## 8. Save checkpoints + results back to Drive (optional)

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# !mkdir -p '/content/drive/MyDrive/2d-chess-detection-artifacts'
# !cp data/checkpoints/classifier.pt '/content/drive/MyDrive/2d-chess-detection-artifacts/'
# !cp data/checkpoints/yolo_runs/board_detector/weights/best.pt '/content/drive/MyDrive/2d-chess-detection-artifacts/yolo_best.pt'
# !cp -r data/output '/content/drive/MyDrive/2d-chess-detection-artifacts/output'